# FENRIX Multi-Company Anonymization Pipeline

**Cloud-first Colab notebook.**  
Collects, anonymizes, and QAs financial data using pre-downloaded SEC archives.

**Requirements:** Google Drive with a SEC filing archive ZIP; GitHub personal access token.

**Outputs:** Anonymized data, manifests, coverage reports, residual QA, export ZIP.

**Security:** Never prints or stores secrets.  Removes credential-bearing remote URLs after cloning.

## 0. Runtime Configuration

Edit the parameters below before running.  NVIDIA is disabled by default.

In [ ]:
# ── COLAB SETUP ───────────────────────────────────────────────────
# These are the only parameters you need to edit.

# Pin the exact commit SHA for deterministic reproducibility.
EXPECTED_COMMIT_SHA = "1c39bf8647e78c7ab5920f96be16863a3cf6d8cc"

REPO_BRANCH = "feature/colab-multicompany-anonymization"
TICKER_LIST = ["NVDA"]  # Add more tickers as needed
YEARS = 10
ENABLE_NVIDIA = False  # Keep disabled unless you have an NVIDIA API key

# Google Drive path to your SEC archive file (e.g., sec_archive.zip)
# Leave empty to skip archive usage and use live SEC only.
DRIVE_SEC_ARCHIVE_PATH = ""  # e.g., "FENRIX/sec_archive.zip"

# SEC source mode: archive-preferred, archive-only, or network-only
SEC_SOURCE_MODE = "archive-preferred"

## 1. Mount Google Drive

Mounts Google Drive for persistent storage.  Outputs and checkpoints are saved here.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

OUTPUT_DRIVE_ROOT = "/content/drive/MyDrive/fenrix_output"
CHECKPOINT_DRIVE_ROOT = "/content/drive/MyDrive/fenrix_checkpoints"


os.makedirs(OUTPUT_DRIVE_ROOT, exist_ok=True)
os.makedirs(CHECKPOINT_DRIVE_ROOT, exist_ok=True)

print("Google Drive mounted. Output root:", OUTPUT_DRIVE_ROOT)

## 2. Read Secrets from Colab Userdata

Secrets are read from Colab's built-in secret manager.  Set these in the Colab UI
under the key icon in the left sidebar (or Runtime → Secrets).

**Required:** `GITHUB_TOKEN`, `SEC_USER_AGENT`

**Optional:** `NVIDIA_API_KEY`, `NVIDIA_MODEL`

Never print or store secret values in notebook output.

In [ ]:
from google.colab import userdata

# Read secrets — never print their values
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
SEC_USER_AGENT = userdata.get("SEC_USER_AGENT")
NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")  # optional
NVIDIA_MODEL = userdata.get("NVIDIA_MODEL")  # optional, used only when NVIDIA is enabled

# Validate required secrets are present
_missing = []
if not GITHUB_TOKEN:
    _missing.append("GITHUB_TOKEN")
if not SEC_USER_AGENT:
    _missing.append("SEC_USER_AGENT")
if _missing:
    raise RuntimeError(f"Missing required secrets: {_missing}. Add them under Colab Secrets.")

print("Secrets loaded: GITHUB_TOKEN [SET], SEC_USER_AGENT [SET]", end="")
if NVIDIA_API_KEY:
    print(", NVIDIA_API_KEY [SET]", end="")
print()

## 3. Clone Repository Securely

Clones the private branch using the GitHub token.  Immediately removes the
credential-bearing remote URL to prevent accidental credential leakage.

In [ ]:
import os
import subprocess

REPO_DIR = "/content/fenrix-synthetic-data"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Scott-Switzer/fenrix-synthetic-data.git"

# Clone or update
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)

# ── IMMEDIATELY remove credential-bearing remote URL ──────────────
subprocess.run(
    [
        "git",
        "remote",
        "set-url",
        "origin",
        "https://github.com/Scott-Switzer/fenrix-synthetic-data.git",
    ],
    cwd=REPO_DIR,
    check=True,
)

os.chdir(REPO_DIR)
%cd {REPO_DIR}

# Verify checkout
result = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True)
print(f"Checked out: {result.stdout.strip()}")

## 4. Install Package and Dependencies

Installs the package with CPU-only dependencies.

In [ ]:
import subprocess

import fenrix_synthetic

# Install fenrix-synthetic package
# Install fenrix-synthetic (use subprocess.run for exit-code enforcement)
result = subprocess.run(["pip", "install", "-e", "."], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"pip install failed: {result.stderr}")
print("Package installed successfully.")

# Install yfinance (runtime dependency)
# Install yfinance
result = subprocess.run(["pip", "install", "yfinance"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"yfinance install failed: {result.stderr}")

# Verify installation
print(f"fenrix-synthetic version: {fenrix_synthetic.__version__}")

## 5. Copy SEC Archive from Drive

Copies the pre-downloaded SEC filing archive from Google Drive to the Colab VM.
The archive is never modified.

In [ ]:
import shutil
from pathlib import Path

SEC_ARCHIVE_PATH = None

if DRIVE_SEC_ARCHIVE_PATH:
    drive_archive = Path("/content/drive/MyDrive") / DRIVE_SEC_ARCHIVE_PATH
    if drive_archive.exists():
        dest = Path("/content/sec_archive_copy")
        dest.mkdir(parents=True, exist_ok=True)
        dest_file = dest / drive_archive.name
        print(
            f"Copying archive from Drive: {drive_archive} ({drive_archive.stat().st_size / 1024 / 1024:.1f} MB)"
        )
        shutil.copy2(drive_archive, dest_file)
        SEC_ARCHIVE_PATH = str(dest_file)
        print(f"Archive copied to: {SEC_ARCHIVE_PATH}")
    else:
        print(f"WARNING: Archive not found at {drive_archive}")
else:
    print("No SEC archive path configured. Will use live SEC collection only.")

## 6. Set Environment Variables

Configures runtime environment.  Never prints secret values.

In [ ]:
import os

os.environ["SEC_USER_AGENT"] = SEC_USER_AGENT
if NVIDIA_API_KEY and ENABLE_NVIDIA:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if NVIDIA_MODEL:
    os.environ["NVIDIA_MODEL"] = NVIDIA_MODEL

print("Environment configured: SEC_USER_AGENT [SET]", end="")
if ENABLE_NVIDIA and NVIDIA_API_KEY:
    print(", NVIDIA_API_KEY [SET]", end="")
print()

## 7. Inventory Archive (if available)

Shows what's in the SEC archive without loading all content into memory.

In [ ]:
import json

if SEC_ARCHIVE_PATH:
    from fenrix_synthetic.collectors.sec_archive import SECArchiveCollector

    collector = SECArchiveCollector(
        archive_path=Path(SEC_ARCHIVE_PATH),
        output_dir=Path("/content/fenrix_run/originals"),
    )
    entries = collector.inventory()
    print(f"Archive files: {len(entries)}")

    # Coverage by ticker/form/year
    coverage = collector.coverage_report()
    print(json.dumps(coverage, indent=2))
else:
    print("No archive to inventory.")

## 8. Run Pipeline

Runs the multi-company pipeline using the package code (no duplicated logic).
Uses SEC archive when available; falls back to live SEC for missing filings
in `archive-preferred` mode.

In [ ]:
from pathlib import Path

from fenrix_synthetic.pipeline.config import PipelineConfig
from fenrix_synthetic.pipeline.runner import PipelineRunner

# Build config
config = PipelineConfig(
    tickers=[{"ticker": t.upper(), "enabled": True, "years": YEARS} for t in TICKER_LIST],
    output_root=Path(OUTPUT_DRIVE_ROOT),
    years=YEARS,
    enable_nvidia=ENABLE_NVIDIA and bool(NVIDIA_API_KEY),
    sec_user_agent=SEC_USER_AGENT,
    sec_archive_path=Path(SEC_ARCHIVE_PATH) if SEC_ARCHIVE_PATH else None,
    sec_source_mode=SEC_SOURCE_MODE,
)

print(f"Run ID: {config.run_id}")
print(f"Tickers: {TICKER_LIST}")
print(f"NVIDIA: {'enabled' if config.enable_nvidia else 'disabled'}")
print(f"SEC source: {config.sec_source_mode}")

# Execute
runner = PipelineRunner(config)
summary = runner.run()

print(f"\nPipeline complete: {summary.get('run_id')}")
for ticker, ts in summary.get("tickers", {}).items():
    status = ts.get("status", "unknown")
    print(f"  {ticker}: {status}")
    for field in ["original_artifacts", "anonymized_artifacts"]:
        if field in ts:
            print(f"    {field}: {ts[field]}")

if "export_zip" in summary:
    print(f"  Export ZIP: {summary['export_zip']}")

## 9. Save Checkpoints to Drive

Copies checkpoints to Google Drive for resumability after runtime restart.

In [ ]:
import shutil


def _copy_outputs_to_drive(src: Path, dst: str) -> None:
    """Copy pipeline outputs to Google Drive for persistence."""
    dst_path = Path(dst)
    dst_path.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        dest_item = dst_path / item.name
        if item.is_dir():
            if dest_item.exists():
                shutil.rmtree(dest_item)
            shutil.copytree(item, dest_item)
        else:
            shutil.copy2(item, dest_item)


# Copy run outputs to Drive
run_dir = config.output_root / config.run_id
if run_dir.exists():
    _copy_outputs_to_drive(run_dir, CHECKPOINT_DRIVE_ROOT + "/" + config.run_id)
    print(f"Outputs saved to Google Drive: {CHECKPOINT_DRIVE_ROOT}/{config.run_id}")
else:
    print("No run directory to save.")

## 10. Resumability

After a runtime restart, re-run cells above and set RESUME_RUN_ID to the previous
run's ID to pick up where you left off.

In [ ]:
# To resume a previous run, set this to the previous run_id:
# RESUME_RUN_ID = "run_20250620_120000"
# Then re-create the PipelineConfig with run_id=RESUME_RUN_ID and resume=True.

print("Resumability is built-in via checkpoints on Google Drive.")
print(f"Current run ID: {config.run_id}")